# Day 12: AdaBoost — "Focus on Your Mistakes" Algorithm 🎯

This notebook implements **AdaBoost (Adaptive Boosting)** in two ways:

1. **From scratch** (using NumPy only) — so you can see exactly how weights, alpha,
   and the final weighted vote are calculated.
2. **Using scikit-learn's `AdaBoostClassifier`** — the production-ready version.

We'll also compare AdaBoost against **Random Forest**, visualize how sample weights
evolve round-by-round, and study the effect of key hyperparameters
(`n_estimators`, `learning_rate`).

---

## 1. Story & Intuition

Imagine you're learning to play basketball. You take 100 shots. A coach analyzes
your performance:

- ✅ 70 shots: You made them → move on.
- ❌ 20 shots: You missed left → practice these more.
- ❌ 10 shots: You missed right → practice these too.

In the next practice session, you spend more time on the shots you previously missed.
Repeat until your weaknesses become strengths. **That is exactly how AdaBoost works.**

> "Random Forest: Everyone votes at once. AdaBoost: Learn from your mistakes,
> one step at a time."

### Random Forest vs AdaBoost

| Random Forest | AdaBoost |
|---|---|
| Trees grow independently | Trees grow sequentially |
| Equal voting | Better trees get more vote |
| Reduces variance | Reduces bias |
| Parallel training | Sequential training |
| Usually deep trees | Usually shallow trees (Decision Stumps) |


In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)
plt.rcParams['figure.figsize'] = (7, 5)


## 2. The Core Idea

- **Weak Learner**: A model that is only slightly better than random guessing
  (e.g. a Decision Stump — a Decision Tree with `depth = 1`).
- **Strong Learner**: A weighted combination of many weak learners that together
  produce high accuracy.

AdaBoost trains weak learners **sequentially**. Each new learner pays more
attention to the samples the previous learners got wrong.


## 3. Building a Decision Stump From Scratch

A Decision Stump is a 1-level decision tree: it picks **one feature** and
**one threshold**, and splits the data into two classes based on whether the
feature is above or below that threshold.

It searches over every feature and every candidate threshold to find the split
that **minimizes the weighted classification error**.


In [ ]:
class DecisionStump:
    """
    A weak learner: single-feature, single-threshold binary classifier.
    Predicts +1 or -1 depending on which side of the threshold a sample falls.
    """

    def __init__(self):
        self.polarity = 1        # 1 or -1: flips which side is "positive"
        self.feature_index = None
        self.threshold = None
        self.alpha = None        # importance of this stump (set later by AdaBoost)

    def predict(self, X):
        n_samples = X.shape[0]
        X_column = X[:, self.feature_index]
        predictions = np.ones(n_samples)

        if self.polarity == 1:
            predictions[X_column < self.threshold] = -1
        else:
            predictions[X_column > self.threshold] = -1

        return predictions


## 4. AdaBoost From Scratch

Following the steps from the guide:

**Step 1 — Assign Equal Weights**
Every training sample starts with weight `1/N`.

**Step 2 — Train a Weak Learner**
Find the stump (feature + threshold + polarity) that minimizes the
*weighted* error.

**Step 3 — Calculate Alpha (α)**

$$\alpha = \frac{1}{2} \ln\left(\frac{1-\text{error}}{\text{error}}\right)$$

| Error | Alpha | Interpretation |
|---|---|---|
| 0.1 | Large positive | Strong learner |
| 0.5 | 0 | No contribution |
| 0.9 | Negative | Prediction should be inverted |

**Step 4 — Update Sample Weights**

- Wrong prediction → `new_weight = old_weight * e^(alpha)`
- Correct prediction → `new_weight = old_weight * e^(-alpha)`

Then normalize all weights so they sum to 1.

**Step 5 — Repeat** for `n_estimators` rounds.

**Step 6 — Final Prediction** (weighted vote)

$$F(x) = \text{sign}\Big(\sum_t \alpha_t \cdot h_t(x)\Big)$$


In [ ]:
class AdaBoostScratch:
    """
    AdaBoost (SAMME, discrete) implemented from scratch with NumPy.
    Labels must be encoded as {-1, +1}.
    """

    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.stumps = []
        # history for visualization / teaching purposes
        self.weight_history = []
        self.error_history = []
        self.alpha_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Step 1: equal weights for every sample
        w = np.full(n_samples, 1 / n_samples)
        self.weight_history.append(w.copy())

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            min_error = float('inf')

            # Step 2: search every feature & threshold for the best weak learner
            for feature_i in range(n_features):
                X_column = X[:, feature_i]
                thresholds = np.unique(X_column)

                for threshold in thresholds:
                    for polarity in (1, -1):
                        predictions = np.ones(n_samples)
                        if polarity == 1:
                            predictions[X_column < threshold] = -1
                        else:
                            predictions[X_column > threshold] = -1

                        # weighted error = sum of weights of misclassified samples
                        misclassified = w[y != predictions]
                        error = misclassified.sum()

                        if error < min_error:
                            min_error = error
                            stump.polarity = polarity
                            stump.threshold = threshold
                            stump.feature_index = feature_i

            # Avoid division by zero / log(0); clip error into a safe range
            EPS = 1e-10
            min_error = np.clip(min_error, EPS, 1 - EPS)

            # Step 3: calculate alpha (scaled by learning_rate)
            alpha = self.learning_rate * 0.5 * np.log((1 - min_error) / min_error)
            stump.alpha = alpha

            # Step 4: update weights
            predictions = stump.predict(X)
            w = w * np.exp(-alpha * y * predictions)   # e^{-alpha} correct, e^{alpha} wrong
            w = w / np.sum(w)                            # normalize

            # bookkeeping for plots later
            self.weight_history.append(w.copy())
            self.error_history.append(min_error)
            self.alpha_history.append(alpha)

            self.stumps.append(stump)

    def predict(self, X):
        # Step 6: weighted vote across all stumps
        stump_preds = np.array([stump.alpha * stump.predict(X) for stump in self.stumps])
        y_pred = np.sum(stump_preds, axis=0)
        return np.sign(y_pred)

    def decision_function(self, X):
        stump_preds = np.array([stump.alpha * stump.predict(X) for stump in self.stumps])
        return np.sum(stump_preds, axis=0)


## 5. Try It On a Toy Dataset

We'll generate a simple 2D binary classification dataset so we can visualize the
decision boundary and watch how AdaBoost improves it round by round.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=300, noise=0.25, random_state=42)
y = np.where(y == 0, -1, 1)   # AdaBoost-from-scratch expects labels in {-1, +1}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#FF6B6B', '#4D96FF']), edgecolor='k')
plt.title("Toy Dataset (make_moons)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


In [ ]:
# Train our from-scratch AdaBoost
model = AdaBoostScratch(n_estimators=50, learning_rate=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"From-scratch AdaBoost test accuracy: {acc:.4f}")


### 5.1 Visualizing the Decision Boundary

As more weak learners (decision stumps) are combined, the overall decision
boundary becomes more complex and accurate — this is the "strong learner"
emerging from many "weak learners".


In [ ]:
def plot_decision_boundary(clf_predict_fn, X, y, title, ax=None):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))

    Z = clf_predict_fn(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    if ax is None:
        fig, ax = plt.subplots()

    ax.contourf(xx, yy, Z, alpha=0.25, cmap=ListedColormap(['#FF6B6B', '#4D96FF']))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#FF6B6B', '#4D96FF']), edgecolor='k', s=20)
    ax.set_title(title)


# Show how the boundary evolves as we add more estimators
rounds_to_show = [1, 3, 10, 50]
fig, axes = plt.subplots(1, len(rounds_to_show), figsize=(20, 4))

for ax, n in zip(axes, rounds_to_show):
    partial_model = AdaBoostScratch(n_estimators=n, learning_rate=1.0)
    partial_model.fit(X_train, y_train)
    plot_decision_boundary(partial_model.predict, X_train, y_train, f"After {n} stump(s)", ax=ax)

plt.tight_layout()
plt.show()


### 5.2 Watching Sample Weights Evolve

Just like the guide's illustration (`A B C D E F G H` bars getting taller for
misclassified points), we can plot how individual sample weights change across
boosting rounds. Misclassified / "hard" samples get heavier, so the next stump
is forced to pay attention to them.


In [ ]:
weights_over_time = np.array(model.weight_history)   # shape: (n_rounds+1, n_samples)

# Track a handful of individual sample weights across rounds
sample_ids = np.random.choice(len(X_train), size=8, replace=False)

plt.figure(figsize=(9, 5))
for sid in sample_ids:
    plt.plot(weights_over_time[:, sid], marker='o', markersize=3, label=f"Sample {sid}")

plt.xlabel("Boosting Round")
plt.ylabel("Sample Weight")
plt.title("How AdaBoost Reweights Samples Over Rounds")
plt.legend(fontsize=8, ncol=2)
plt.show()


### 5.3 Error and Alpha per Round

- `error_history`: the weighted error of the best stump found in each round.
- `alpha_history`: the resulting importance (α) assigned to that stump.

Notice that as error decreases, alpha increases — a rule directly from the
formula α = ½·ln((1−error)/error).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(model.error_history, color='#FF6B6B', marker='o', markersize=3)
axes[0].set_title("Weighted Error per Round")
axes[0].set_xlabel("Round")
axes[0].set_ylabel("Error")
axes[0].axhline(0.5, color='gray', linestyle='--', label='error = 0.5 (no skill)')
axes[0].legend()

axes[1].plot(model.alpha_history, color='#4D96FF', marker='o', markersize=3)
axes[1].set_title("Alpha (Learner Importance) per Round")
axes[1].set_xlabel("Round")
axes[1].set_ylabel("Alpha")

plt.tight_layout()
plt.show()


## 6. AdaBoost With Scikit-learn

Now let's use the production-ready `AdaBoostClassifier` and confirm our
from-scratch implementation is directionally correct.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# sklearn expects standard {0, 1} labels
y_train_sklearn = np.where(y_train == -1, 0, 1)
y_test_sklearn = np.where(y_test == -1, 0, 1)

sklearn_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision Stump
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

sklearn_ada.fit(X_train, y_train_sklearn)
sklearn_pred = sklearn_ada.predict(X_test)
sklearn_acc = accuracy_score(y_test_sklearn, sklearn_pred)

print(f"Scikit-learn AdaBoost test accuracy: {sklearn_acc:.4f}")
print(f"From-scratch AdaBoost test accuracy: {acc:.4f}")


## 7. AdaBoost vs Random Forest

Let's directly compare the two ensemble strategies on the same dataset:
sequential bias-reduction (AdaBoost) vs parallel variance-reduction (Random Forest).


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

results = {}

# AdaBoost
start = time.time()
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
)
ada.fit(X_train, y_train_sklearn)
ada_time = time.time() - start
results["AdaBoost"] = {
    "accuracy": accuracy_score(y_test_sklearn, ada.predict(X_test)),
    "train_time_sec": ada_time
}

# Random Forest
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train_sklearn)
rf_time = time.time() - start
results["Random Forest"] = {
    "accuracy": accuracy_score(y_test_sklearn, rf.predict(X_test)),
    "train_time_sec": rf_time
}

import pandas as pd
pd.DataFrame(results).T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_decision_boundary(lambda X: ada.predict(X), X_train, np.where(y_train==-1,0,1),
                        "AdaBoost Decision Boundary", ax=axes[0])
plot_decision_boundary(lambda X: rf.predict(X), X_train, np.where(y_train==-1,0,1),
                        "Random Forest Decision Boundary", ax=axes[1])

plt.tight_layout()
plt.show()


## 8. Important Hyperparameters

| Hyperparameter | Description | Recommended Value |
|---|---|---|
| `estimator` | Weak learner | Decision Tree (`depth=1`) |
| `n_estimators` | Number of boosting rounds | 50–500 |
| `learning_rate` | Controls contribution of each learner | 0.1–1.0 |
| `algorithm` | Boosting algorithm | `SAMME` / `SAMME.R` |

**Trade-off:**
- Lower `learning_rate` → needs more `n_estimators`.
- Higher `learning_rate` → needs fewer `n_estimators`.
- A common combination: `learning_rate=0.1`, `n_estimators=500`.

Let's verify this trade-off empirically.


In [ ]:
from sklearn.model_selection import cross_val_score

learning_rates = [0.01, 0.1, 0.5, 1.0]
n_estimators_range = [10, 25, 50, 100, 200, 500]

plt.figure(figsize=(9, 6))

for lr in learning_rates:
    scores = []
    for n_est in n_estimators_range:
        clf = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
        )
        cv_scores = cross_val_score(clf, X_train, y_train_sklearn, cv=5)
        scores.append(cv_scores.mean())
    plt.plot(n_estimators_range, scores, marker='o', label=f"learning_rate={lr}")

plt.xlabel("n_estimators")
plt.ylabel("Cross-validated Accuracy")
plt.title("Learning Rate vs n_estimators Trade-off")
plt.legend()
plt.show()


## 9. Real-World Example: Medical Diagnosis Dataset

AdaBoost is commonly used for tasks like **medical diagnosis**, **spam
detection**, and **face detection**. Let's apply it to the classic
Breast Cancer Wisconsin dataset (binary classification: malignant vs benign).


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc
)

scaler = StandardScaler()
X_bc_train = scaler.fit_transform(X_bc_train)
X_bc_test = scaler.transform(X_bc_test)

ada_bc = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
)
ada_bc.fit(X_bc_train, y_bc_train)

bc_acc = accuracy_score(y_bc_test, ada_bc.predict(X_bc_test))
print(f"AdaBoost accuracy on Breast Cancer dataset: {bc_acc:.4f}")


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print(classification_report(y_bc_test, ada_bc.predict(X_bc_test), target_names=data.target_names))

ConfusionMatrixDisplay.from_estimator(ada_bc, X_bc_test, y_bc_test, display_labels=data.target_names, cmap='Blues')
plt.title("AdaBoost Confusion Matrix — Breast Cancer Dataset")
plt.show()


### Feature Importance

Since AdaBoost combines many decision stumps, we can also inspect which
features were most frequently/strongly used to split the data.


In [ ]:
importances = ada_bc.feature_importances_
feat_names = data.feature_names

order = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(9, 5))
plt.barh(np.array(feat_names)[order][::-1], importances[order][::-1], color='#4D96FF')
plt.title("Top 10 Feature Importances — AdaBoost")
plt.xlabel("Importance")
plt.show()


## 10. AdaBoost vs Random Forest — Summary Table

| Feature | AdaBoost | Random Forest |
|---|---|---|
| Training | Sequential | Parallel |
| Tree Type | Shallow Stumps | Deep Trees |
| Tree Importance | Weighted | Equal |
| Main Goal | Reduce Bias | Reduce Variance |
| Training Speed | Slower | Faster |
| Noise Sensitivity | High | Lower |

### When to Use AdaBoost
**Suitable for:** simple classification problems, boosting weak learners,
small–medium datasets, learning boosting concepts.

**Avoid when:** very noisy datasets, many outliers, high-dimensional sparse
data, or when you need highly parallel training.

### Common Problems

| Problem | Cause |
|---|---|
| Overfitting | Focuses too much on noisy samples |
| Outlier Sensitivity | Outliers receive very high weights |
| Slow Training | Sequential learning |
| Class Imbalance | Majority class dominates |

### Real-World Applications
- Face Detection
- Text Classification
- Medical Diagnosis
- Spam Detection
- Customer Risk Prediction


## 11. Interview Questions (Quick Answers)

1. **What is AdaBoost?** A boosting ensemble method that sequentially combines
   weak learners, giving more weight to misclassified samples at each step.
2. **What is a Weak Learner?** A model slightly better than random guessing
   (commonly a decision stump).
3. **Why are Decision Stumps commonly used?** They're fast to train and simple,
   letting AdaBoost focus its power on the boosting process rather than
   individual model complexity.
4. **How are sample weights updated?** Increased for misclassified samples
   (`× e^α`), decreased for correctly classified ones (`× e^(-α)`), then
   normalized.
5. **What is Alpha (α)?** The weight/importance given to a weak learner in the
   final vote, based on its error rate.
6. **Why "Adaptive" Boosting?** Because it adapts by reweighting samples based
   on previous errors, rather than treating all rounds identically.
7. **AdaBoost vs Random Forest?** Sequential vs parallel training; bias
   reduction vs variance reduction; weighted vs equal voting.
8. **Role of Learning Rate?** Shrinks each stump's contribution, trading off
   against the number of estimators needed.
9. **What if error > 0.5?** Alpha becomes negative — meaning the learner's
   predictions are inverted (worse than random, so we flip them).
10. **Limitations of AdaBoost?** Sensitive to noisy data and outliers; slower
    (sequential) training; not ideal for very high-dimensional sparse data.


## 12. Conclusion

AdaBoost demonstrates a simple but powerful idea: **learn from your mistakes,
one weak learner at a time.** By combining many simple models — each focused on
the errors of the ones before it — AdaBoost builds a strong, accurate
classifier while keeping individual components lightweight and interpretable.

**Key takeaways:**
- Weight *samples*, not just aggregate predictions like Random Forest.
- Each learner's vote is scaled by its own accuracy (`alpha`).
- Great for reducing bias on clean data; be cautious with noisy/outlier-heavy data.

Happy Learning! 🎯
